## NLP Basics Updated

Review class slides before starting this. Let's just look at some NLP basics to get started. Note that we'll use the [https://www.nltk.org/](https://www.nltk.org/) library.

In [ ]:
# some of the basics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# we'll need this to do nlp
import nltk

# we'll need this to remove stopwords
from nltk.corpus import stopwords
import string

# this will help us transform a corpus
from sklearn.feature_extraction.text import CountVectorizer

# to generate wordcloud
# from wordcloud import WordCloud, STOPWORDS

# combine this data with a model we already know
from sklearn.cluster import KMeans

# tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer

# for sentiment analysis
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.stem import SnowballStemmer

### Demonstration 1: Basics

Below is a course description. How can we break this up to analyze it?  We'll convert it to tokens and analyze the frequency of the tokens.

In [ ]:
sentence = "Study designed to prepare the student to integrate systems of diverse architecture so that they share information and resources efficiently. Introduction to the basic system programming concepts required to perform these tasks in multiple environments. Skills learned include computer management such as disk and memory management, file handling, system security, and customization of the user interface on various operating systems. Other skills include file formatting, transmission, translation and scripting to enable information sharing across multiple platforms. Programs are implemented in multiple operating systems' command languages."

In [ ]:
# break the sentence into tokens
tokens = nltk.word_tokenize(sentence)

What is tokens? 

In [ ]:
# to do: explore what tokens is


In [ ]:
# find the frequency distribution of tokens in the sentence
frequency_distribution = nltk.FreqDist(tokens)
frequency_distribution

In [ ]:
# note the type of data this gives us back
type(frequency_distribution)

We can consult the documentation in order to determine what to do with this object: https://www.nltk.org/api/nltk.probability.FreqDist.html

One of the things we can do is use the object's plot() function to see the distribution.

In [ ]:
# plot it!
frequency_distribution.max()

In [ ]:
frequency_distribution.freq("to")

In [ ]:
frequency_distribution.plot(10)

We'll note that the words showing up more often are common words that we want to exclude. We can use the stopwords library to remove these words.

In [ ]:
# Get a list of stop words in English
stop_words = set(stopwords.words("english"))

In [ ]:
# remove punctuation and stop words
tokens = [token for token in tokens if token not in string.punctuation]
filtered_words = [token for token in tokens if token.lower() not in stop_words]

In [ ]:
# Let's now see the updated frequency distribution
frequency_distribution = nltk.FreqDist(filtered_words)

In [ ]:
# to do: plot it!

Now let's also perform stemming and see what impact this has. 

In [ ]:
stemmer = SnowballStemmer(language='english')

In [ ]:
stemmed_words = [stemmer.stem(word) for word in filtered_words]

In [ ]:
frequency_distribution = nltk.FreqDist(stemmed_words)

In [ ]:
frequency_distribution.plot(10)

#### Next: Representing course descriptions as "bag-of-words"

In [ ]:
# our corpus -- multiple course descriptions
documents = [
    "Study and application of the elements of proper design to create gaming and simulation applications using the chosen programming paradigm. Coursework includes using favored industry standard programming languages, such as C++ and C#, to write sample game and simulation applications. Study and analysis of the reasons a specific language is more suitable for gaming application.",
    "Study designed to prepare the student to integrate systems of diverse architecture so that they share information and resources efficiently. Introduction to the basic system programming concepts required to perform these tasks in multiple environments. Skills learned include computer management such as disk and memory management, file handling, system security, and customization of the user interface on various operating systems. Other skills include file formatting, transmission, translation and scripting to enable information sharing across multiple platforms. Programs are implemented in multiple operating systems' command languages.",
    "Study of the systems analysis process used to design solutions to business problems. The phases of the System Development Life Cycle (SDLC), up to and including requirements analysis, are presented. Emphasis on teamwork in addressing problem identification, data gathering, and project management. Course work includes using the SDLC to solve an actual business problem; identifying requirements through interviews, questionnaires and research; and using the rapid application development process (RAD) to demonstrate human/computer interfaces. Project management techniques are used to schedule activities and evaluate progress. The final product will be a system requirements document and presentation to the appropriate stakeholders.",
    "Introduction to the administration of computer networks, including installing and configuring servers and clients; creating and maintaining secure network resources and accounts using standard naming conventions; developing documentation; and performing basic troubleshooting and performance analyses. ",
    "A study of simulation as it relates to applications that use game engines or related programming environments to simulate a concept, behavior, or interaction. Students will recognize the role of human computer interactions (HCI) in the implementation of effective simulations. Introduction to various actual simulations on and off-campus. The course work includes use of a simulation development tool, algorithms, and structures to create simulation applications."
]

In [ ]:
# Create a CountVectorizer object
vectorizer = CountVectorizer()

In [ ]:
# Fit and transform the documents
X = vectorizer.fit_transform(documents)

In [ ]:
# Get the feature names (vocabulary)
feature_names = vectorizer.get_feature_names_out()

In [ ]:
# Convert results to a dataframe because we like to work with them
df = pd.DataFrame(data=X.toarray(), columns=feature_names)

In [ ]:
df

Use clustering to find similar documents

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=5)
kmeans.fit(df)
kmeans.labels_

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

How did it do?

### Demonstration 2: TF-IDF

We'll use the same corpus as above.  But instead we'll create a TfidfVectorizer() object and use it to transform our corpus. 

See https://scikit-learn.org/1.5/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html for more details.

In [ ]:
# Create a tf-idf vector object - remove english stop words
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform the documents
result = tfidf.fit_transform(documents)  # document is just our list of documents/course descriptions


What does this produce? We see the feature names.

In [ ]:
# extracting feature names
tfidf_tokens = tfidf.get_feature_names_out()
tfidf_tokens

Let's put it all together.
The words with the highest values are the more unique to particular documents.

In [ ]:
df_idf = pd.DataFrame(tfidf.idf_, index=tfidf.get_feature_names_out(),columns=["idf_weights"])
df_idf.sort_values(by=['idf_weights'], ascending=False).head(25)

The words with the lowest values are the less unique to a particular document.

In [ ]:
df_idf.sort_values(by=['idf_weights']).head(10)

In [ ]:
result.toarray()

In [ ]:
df_result = pd.DataFrame(
    data=result.toarray(), 
    index=["Doc1", "Doc2", "Doc3", "Doc4", "Doc5"], 
    columns=tfidf_tokens
)
pd.set_option('display.max_columns', None)
df_result

In [ ]:
max_values_per_row = df_result.max(axis=1)
max_values_per_row = df_result.idxmax(axis=1)
max_values_per_row

### Demonstration 3: Sentiment

Now we'll take a look at how to determine sentiment using the Vader library from nltk.

In [ ]:
# text to analyze
text = "I love to not hate Python!"

We can create a SentimentIntensityAnalyzer object to determine the sentiment. This is designed to get the sentiment score from a sentence.
See https://www.nltk.org/api/nltk.sentiment.vader.html#nltk.sentiment.vader.SentimentIntensityAnalyzer to learn more. 

In [ ]:
# create the object
analyzer = SentimentIntensityAnalyzer()

In [ ]:
# pass it a sentence to analyze and it returns a dictionary of the scores
scores = analyzer.polarity_scores(text)
type(scores)

In [ ]:
# placing the scores in a dataframe for easier processing
scores_df = pd.DataFrame(scores, index=[0])
scores_df

In [ ]:
# plotting the scores
ax = sns.barplot(data=scores_df)
ax.set_title('Sentiment Scores')
plt.show()

#### Now you try

In [ ]:
reviews_df = pd.read_csv("restaurant_reviews.csv")
reviews_df

While this is an already labeled dataset, we are going to play around with determining the sentiment. 

What is the general sentiment for "The Frog at Bercy Village"?

In [ ]:
# to do: get just that data
# call it frog_df

In [ ]:
# to do: look at the value counts


Reviews look positive. Let's do some prep of the reviews before processing. We'll remove stop words. 

In [ ]:
# note that vader removes stopwords for us, so we don't really need to do this. But this is good practice.
def clean_review(review):
    tokens = review.split()
    filtered_words = [token for token in tokens if token.lower() not in stop_words]
    cleaned_words = " ".join(filtered_words)
    return cleaned_words

frog_df['Review'] = frog_df['Review'].apply(clean_review)

In [ ]:
frog_df['Review']

Determine sentiment and add as a new features in dataframe.

In [ ]:
# create sentiment analyzer object
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(review):
    # to do: determine scores & return the "compound" value
    # hint: use the analyzer to get the polarity score of a review
    
    return scores["compound"]
    # return "Positive" if scores["compound"] >= 0.5 else "Neutral" if scores["compound"] >-0.5 else "Negative"

frog_df['sentiment_calc'] = frog_df['Review'].apply(get_sentiment)

In [ ]:
frog_df.head(30)

What is the average sentiment?

In [ ]:
# to do: get mean


### Demonstration 4: Topic Analysis

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

For more information on the above, see https://scikit-learn.org/1.5/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html
    

In [ ]:
# get top 10 topics
lda_model = LatentDirichletAllocation(n_components=10, learning_method='online', random_state=42, max_iter=1) 

In [ ]:
# fit and transform
lda_topics=lda_model.fit_transform(result)

In [ ]:
topic_words = lda_model.components_
topic_words.shape

In [ ]:
# number of topics for each document
n = 5

# walk through each topic with most important words first
for i, topic_dist in enumerate(topic_words):
    sorted_topic_dist = np.argsort(topic_dist)
    topic_words = np.array(tfidf.get_feature_names())[sorted_topic_dist]
    topic_words = topic_words[:-n:-1] # start at end only & only get last 4
    print("Topic", str(i+1), topic_words)